# Лабораторная работа № 3

## Профилирование и качество данных: raw → staging

**Набор данных:** Seoul Bike Sharing Demand.

**Аналитический вопрос:** как зависит спрос на велосипеды от температуры в разные часы суток и сезоны?

**Гранулярность:** одна запись за один час конкретной даты. **Бизнес-ключ:** `Date` и `Hour`. Для анализа используются число аренд, температура, время и сезон. Часы, когда прокат не работал, отмечаются отдельным признаком.

## 1. Исходные данные

Исходный CSV читается из бакета `raw` через DuckDB. Для чтения заданы 14 имён полей, поскольку заголовок CSV содержит нестандартные символы.

In [1]:
import os
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Откройте ноутбук внутри репозитория itad-labs.")
load_dotenv(PROJECT_ROOT / ".env", override=True)
for name in ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY"):
    if not os.environ.get(name):
        raise ValueError(f"В .env отсутствует {name}")

raw_path = "s3://raw/seoul-bike-sharing-demand/ingested_on=2026-09-30/SeoulBikeData.csv"
dataset_slug = "seoul-bike-sharing-demand"
if not raw_path.startswith(f"s3://raw/{dataset_slug}/ingested_on="):
    raise ValueError("Неверный raw-путь или slug")
source_ingested_on = raw_path.split("/ingested_on=", 1)[1].split("/", 1)[0]
output_prefix = f"s3://staging/{dataset_slug}/source_ingested_on={source_ingested_on}"

def sqlstr(value):
    return "'" + str(value).replace("'", "''") + "'"

def show_table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in row) + "</tr>"
                   for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"SET s3_endpoint = {sqlstr(os.environ['S3_ENDPOINT'].removeprefix('http://').removeprefix('https://'))}")
con.execute(f"SET s3_access_key_id = {sqlstr(os.environ['S3_ACCESS_KEY'])}")
con.execute(f"SET s3_secret_access_key = {sqlstr(os.environ['S3_SECRET_KEY'])}")
con.execute("SET s3_url_style = 'path'")
con.execute("SET s3_use_ssl = false")

source_columns = [
    "Date", "Rented Bike Count", "Hour", "Temperature(°C)", "Humidity(%)",
    "Wind speed (m/s)", "Visibility (10m)", "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)", "Rainfall(mm)", "Snowfall (cm)",
    "Seasons", "Holiday", "Functioning Day",
]
column_list = ", ".join(sqlstr(name) for name in source_columns)
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW raw_data AS
    SELECT * FROM read_csv(
        {sqlstr(raw_path)}, header = false, skip = 1,
        names = [{column_list}], auto_detect = true, all_varchar = true
    )
""")
show_table(["Raw CSV", "Выходной префикс"], [(raw_path, output_prefix)])
show_table(["Date", "Hour", "Rented Bike Count", "Temperature", "Seasons", "Functioning Day"],
           con.execute("""SELECT "Date", "Hour", "Rented Bike Count", "Temperature(°C)",
                                "Seasons", "Functioning Day" FROM raw_data LIMIT 5""").fetchall())

Raw CSV,Выходной префикс
s3://raw/seoul-bike-sharing-demand/ingested_on=2026-09-30/SeoulBikeData.csv,s3://staging/seoul-bike-sharing-demand/source_ingested_on=2026-09-30


Date,Hour,Rented Bike Count,Temperature,Seasons,Functioning Day
01/12/2017,0,254,-5.2,Winter,Yes
01/12/2017,1,204,-5.5,Winter,Yes
01/12/2017,2,173,-6,Winter,Yes
01/12/2017,3,107,-6.2,Winter,Yes
01/12/2017,4,78,-6,Winter,Yes


## 2. Профиль данных

Проверяются `Date`, `Hour`, `Rented Bike Count`, `Temperature(°C)`, `Seasons` и `Functioning Day`. Пропусками считаются `NULL` и пустые строки.

In [2]:
raw_count = con.execute("SELECT count(*) FROM raw_data").fetchone()[0]
schema = {row[0]: row[1] for row in con.execute("DESCRIBE raw_data").fetchall()}
show_table(["Строк", "Полей DuckDB"], [(raw_count, len(schema))])

main_fields = ("Date", "Hour", "Rented Bike Count", "Temperature(°C)", "Seasons")
profile_fields = (*main_fields, "Functioning Day")
profile_rows = []
for field in profile_fields:
    quoted = '"' + field.replace('"', '""') + '"'
    missing = con.execute(f"""
        SELECT count(*) FROM raw_data
        WHERE {quoted} IS NULL OR trim(CAST({quoted} AS VARCHAR)) = ''
    """).fetchone()[0]
    examples = con.execute(f"""
        SELECT DISTINCT {quoted} FROM raw_data
        WHERE {quoted} IS NOT NULL AND trim(CAST({quoted} AS VARCHAR)) <> ''
        ORDER BY 1 LIMIT 2
    """).fetchall()
    profile_rows.append((field, schema[field], ", ".join(str(r[0]) for r in examples),
                         raw_count, missing, f"{missing / raw_count:.2%}" if raw_count else "—"))
show_table(["Поле", "Тип в raw", "Примеры", "Проверено", "Пропусков", "Доля"], profile_rows)

Строк,Полей DuckDB
8760,15


Поле,Тип в raw,Примеры,Проверено,Пропусков,Доля
Date,VARCHAR,"01/01/2018, 01/02/2018",8760,0,0.00%
Hour,VARCHAR,"0, 1",8760,0,0.00%
Rented Bike Count,VARCHAR,"0, 10",8760,0,0.00%
Temperature(°C),VARCHAR,"-0.1, -0.2",8760,0,0.00%
Seasons,VARCHAR,"Autumn, Spring",8760,0,0.00%
Functioning Day,VARCHAR,"No, Yes",8760,0,0.00%


### Приведение типов

Дата преобразуется из формата `%d/%m/%Y`; числовые значения приводятся с помощью `TRY_CAST`. Непреобразуемые значения становятся `NULL` и учитываются в проверках.

In [3]:
con.execute("""
CREATE OR REPLACE TEMP VIEW prepared_data AS
SELECT r.*,
       CAST(TRY_STRPTIME(NULLIF(TRIM("Date"), ''), '%d/%m/%Y') AS DATE) AS event_date,
       TRY_CAST(NULLIF(TRIM("Hour"), '') AS INTEGER) AS event_hour,
       TRY_CAST(NULLIF(TRIM("Rented Bike Count"), '') AS BIGINT) AS rented_bike_count,
       TRY_CAST(NULLIF(TRIM("Temperature(°C)"), '') AS DOUBLE) AS temperature_c,
       NULLIF(TRIM("Seasons"), '') AS season_clean,
       NULLIF(TRIM("Functioning Day"), '') AS functioning_day_clean
FROM raw_data AS r
""")

first_last = con.execute("SELECT min(event_date), max(event_date) FROM prepared_data").fetchone()
key_counts = con.execute("""
    SELECT count(*) AS checked, count(*) -
           (SELECT count(*) FROM (
                SELECT DISTINCT event_date, event_hour FROM prepared_data
                WHERE event_date IS NOT NULL AND event_hour BETWEEN 0 AND 23
            )) AS excessive
    FROM prepared_data
    WHERE event_date IS NOT NULL AND event_hour BETWEEN 0 AND 23
""").fetchone()
show_table(["Мин. дата", "Макс. дата", "Проверено ключей", "Избыточных повторов Date + Hour"],
           [(*first_last, *key_counts)])
show_table(["Сезон", "Количество строк"], con.execute("""
    SELECT "Seasons", count(*) FROM raw_data GROUP BY "Seasons" ORDER BY "Seasons"
""").fetchall())
show_table(["Час", "Количество строк"], con.execute("""
    SELECT event_hour, count(*) FROM prepared_data GROUP BY event_hour ORDER BY event_hour
""").fetchall())

Мин. дата,Макс. дата,Проверено ключей,Избыточных повторов Date + Hour
2017-12-01,2018-11-30,8760,0


Сезон,Количество строк
Autumn,2184
Spring,2208
Summer,2208
Winter,2160


Час,Количество строк
0,365
1,365
2,365
3,365
4,365
5,365
6,365
7,365
8,365
9,365


## 3. Правила качества

Для всех правил порог составляет 0 нарушивших строк. В коде пороги заданы переменными; превышение определяется условием `violations > threshold`.

In [4]:
con.execute("""
CREATE OR REPLACE TEMP VIEW checked_data AS
WITH base AS (
    SELECT *,
           CASE
               WHEN EXTRACT(MONTH FROM event_date) IN (12, 1, 2) THEN 'Winter'
               WHEN EXTRACT(MONTH FROM event_date) IN (3, 4, 5) THEN 'Spring'
               WHEN EXTRACT(MONTH FROM event_date) IN (6, 7, 8) THEN 'Summer'
               WHEN EXTRACT(MONTH FROM event_date) IN (9, 10, 11) THEN 'Autumn'
           END AS expected_season,
           COUNT(*) OVER (PARTITION BY event_date, event_hour) AS date_hour_occurrences
    FROM prepared_data
)
SELECT *,
    (event_date IS NOT NULL AND event_hour BETWEEN 0 AND 23
     AND date_hour_occurrences > 1) AS bad_duplicate_key,
    (event_date IS NULL) AS bad_date,
    (event_hour IS NULL OR event_hour NOT BETWEEN 0 AND 23
     OR NOT COALESCE(REGEXP_FULL_MATCH(TRIM("Hour"), '[0-9]{1,2}'), FALSE)) AS bad_hour,
    (rented_bike_count IS NULL OR rented_bike_count < 0
     OR NOT COALESCE(REGEXP_FULL_MATCH(TRIM("Rented Bike Count"), '[0-9]+'), FALSE)) AS bad_count,
    (temperature_c IS NULL OR NOT ISFINITE(temperature_c)) AS bad_temperature,
    (season_clean IS NULL OR season_clean NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
     OR (event_date IS NOT NULL AND season_clean IS DISTINCT FROM expected_season)) AS bad_season,
    (functioning_day_clean IS NULL OR functioning_day_clean NOT IN ('Yes', 'No')) AS bad_functioning,
    COALESCE(functioning_day_clean = 'No', FALSE) AS flag_non_functioning
FROM base
""")

rule_results = {}

def record_rule(rule_id, checked, violated, maximum, severity, action):
    checked, violated = int(checked), int(violated)
    if not (isinstance(maximum, int) and maximum >= 0):
        raise ValueError("Порог должен быть целым неотрицательным числом")
    result = (rule_id, checked, violated,
              f"{violated / checked:.2%}" if checked else "—",
              f"count ≤ {maximum}", "Превышен" if violated > maximum else "Не превышен",
              severity, action)
    rule_results[rule_id] = result
    show_table(["ID", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Критичность", "Действие"], [result])

### 3.1. `unique_date_hour`

- **Поля:** `Date`, `Hour`; проверяются записи с корректной датой и часом 0–23.
- **Условие:** сочетание даты и часа должно быть уникальным. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine` для всех записей повторяющегося ключа.
- **Значение:** дубликаты искажают почасовой спрос.

In [5]:
max_duplicate_keys = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_duplicate_key)
    FROM checked_data
    WHERE event_date IS NOT NULL AND event_hour BETWEEN 0 AND 23
""").fetchone()
record_rule("unique_date_hour", checked, violated, max_duplicate_keys, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
unique_date_hour,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.2. `valid_date`

- **Поле:** `Date`; проверяются все строки.
- **Условие:** дата заполнена и соответствует формату `%d/%m/%Y`. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine`.
- **Значение:** дата необходима для определения времени и сезона.

In [6]:
max_bad_dates = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_date)
    FROM checked_data
""").fetchone()
record_rule("valid_date", checked, violated, max_bad_dates, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
valid_date,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.3. `valid_hour`

- **Поле:** `Hour`; проверяются все строки.
- **Условие:** целое число от 0 до 23. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine`.
- **Значение:** некорректный час искажает почасовое распределение спроса.

In [7]:
max_bad_hours = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_hour)
    FROM checked_data
""").fetchone()
record_rule("valid_hour", checked, violated, max_bad_hours, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
valid_hour,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.4. `nonnegative_rentals`

- **Поле:** `Rented Bike Count`; проверяются все строки.
- **Условие:** заполненное неотрицательное целое число. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine`.
- **Значение:** количество аренд используется как показатель спроса.

In [8]:
max_bad_rentals = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_count)
    FROM checked_data
""").fetchone()
record_rule("nonnegative_rentals", checked, violated, max_bad_rentals, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
nonnegative_rentals,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.5. `valid_temperature`

- **Поле:** `Temperature(°C)`; проверяются все строки.
- **Условие:** заполненное конечное числовое значение. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine`.
- **Значение:** температура необходима для анализа спроса.

In [9]:
max_bad_temperatures = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_temperature)
    FROM checked_data
""").fetchone()
record_rule("valid_temperature", checked, violated, max_bad_temperatures, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
valid_temperature,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.6. `season_matches_date`

- **Поля:** `Seasons`, `Date`; проверяются все строки, сопоставление выполняется при корректной дате.
- **Условие:** `Winter` (12–2), `Spring` (3–5), `Summer` (6–8) или `Autumn` (9–11), в соответствии с месяцем даты. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine`.
- **Значение:** ошибки сезона искажают сравнение спроса по временам года.

In [10]:
max_bad_seasons = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_season)
    FROM checked_data
""").fetchone()
record_rule("season_matches_date", checked, violated, max_bad_seasons, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
season_matches_date,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.7. `valid_functioning_day`

- **Поле:** `Functioning Day`; проверяются все строки.
- **Условие:** значение `Yes` или `No`. **Порог:** 0 строк.
- **Критичность:** `critical`. **Действие:** `quarantine`.
- **Значение:** по этому полю выделяются часы работы проката.

In [11]:
max_bad_functioning = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE bad_functioning)
    FROM checked_data
""").fetchone()
record_rule("valid_functioning_day", checked, violated, max_bad_functioning, "critical", "quarantine")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
valid_functioning_day,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine


### 3.8. `functioning_day_for_analysis`

- **Поле:** `Functioning Day`; проверяются строки со значениями `Yes` или `No`.
- **Условие:** для дальнейшего анализа подходят часы со статусом `Yes`. **Порог:** 0 строк.
- **Критичность:** `info`. **Действие:** `flag` при `No`.
- **Значение:** нерабочие часы сохраняются в данных, но помечаются для последующей фильтрации.

In [12]:
max_non_functioning = 0
checked, violated = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE flag_non_functioning)
    FROM checked_data
    WHERE functioning_day_clean IN ('Yes', 'No')
""").fetchone()
record_rule("functioning_day_for_analysis", checked, violated, max_non_functioning, "info", "flag")

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
functioning_day_for_analysis,8760,295,3.37%,count ≤ 0,Превышен,info,flag


### Сводная таблица проверок

In [13]:
show_table(["ID", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Критичность", "Действие"],
           rule_results.values())

ID,Проверено,Нарушений,Доля,Порог,Сравнение,Критичность,Действие
unique_date_hour,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
valid_date,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
valid_hour,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
nonnegative_rentals,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
valid_temperature,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
season_matches_date,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
valid_functioning_day,8760,0,0.00%,count ≤ 0,Не превышен,critical,quarantine
functioning_day_for_analysis,8760,295,3.37%,count ≤ 0,Превышен,info,flag


## 4. Подготовка staging и карантина

В `staging` сохраняются записи без критических нарушений с приведёнными типами полей. Признак `quality_non_functioning_hour_violated` равен `TRUE` для нерабочих часов.

В карантине сохраняются исходные поля, `quality_rule_ids` и `quarantine_reason`. Идентификаторы нескольких нарушений разделяются символом `;`.

In [14]:
crit = ("bad_duplicate_key OR bad_date OR bad_hour OR bad_count OR "
        "bad_temperature OR bad_season OR bad_functioning")
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT event_date, event_hour AS hour_of_day,
           rented_bike_count, temperature_c,
           season_clean AS season, functioning_day_clean AS functioning_day,
           flag_non_functioning AS quality_non_functioning_hour_violated
    FROM checked_data
    WHERE NOT ({crit})
""")

original_projection = ", ".join('"' + col.replace('"', '""') + '"' for col in source_columns)
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT {original_projection},
           CONCAT(
             CASE WHEN bad_duplicate_key THEN 'unique_date_hour;' ELSE '' END,
             CASE WHEN bad_date THEN 'valid_date;' ELSE '' END,
             CASE WHEN bad_hour THEN 'valid_hour;' ELSE '' END,
             CASE WHEN bad_count THEN 'nonnegative_rentals;' ELSE '' END,
             CASE WHEN bad_temperature THEN 'valid_temperature;' ELSE '' END,
             CASE WHEN bad_season THEN 'season_matches_date;' ELSE '' END,
             CASE WHEN bad_functioning THEN 'valid_functioning_day;' ELSE '' END
           ) AS quality_rule_ids,
           CONCAT(
             CASE WHEN bad_duplicate_key THEN 'Повтор Date + Hour; ' ELSE '' END,
             CASE WHEN bad_date THEN 'Неверная дата; ' ELSE '' END,
             CASE WHEN bad_hour THEN 'Неверный час; ' ELSE '' END,
             CASE WHEN bad_count THEN 'Некорректное число аренд; ' ELSE '' END,
             CASE WHEN bad_temperature THEN 'Некорректная температура; ' ELSE '' END,
             CASE WHEN bad_season THEN 'Некорректный сезон; ' ELSE '' END,
             CASE WHEN bad_functioning THEN 'Некорректный статус работы; ' ELSE '' END
           ) AS quarantine_reason
    FROM checked_data
    WHERE {crit}
""")
counts = con.execute("""
SELECT (SELECT count(*) FROM raw_data),
       (SELECT count(*) FROM staging_data),
       (SELECT count(*) FROM quarantine_data)
""").fetchone()
assert counts[0] == counts[1] + counts[2], "Не сошёлся баланс raw = staging + карантин"
show_table(["Raw", "Staging", "Карантин"], [counts])
show_table(["Дата", "Час", "Аренд", "Температура", "Сезон", "Нерабочий час"],
           con.execute("SELECT * FROM staging_data ORDER BY event_date, hour_of_day LIMIT 5").fetchall())
show_table(["Исходная дата", "Час", "Причины", "Пояснение"],
           con.execute("""SELECT "Date", "Hour", quality_rule_ids, quarantine_reason
                          FROM quarantine_data LIMIT 3""").fetchall())

Raw,Staging,Карантин
8760,8760,0


Исходная дата,Час,Причины,Пояснение


### Сохранение результатов в S3

Результаты записываются в `staging.parquet` и `quarantine.parquet` с датой загрузки исходного файла.

In [15]:
endpoint = os.environ["S3_ENDPOINT"]
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
for name in ("staging", "quarantine"):
    target = f"{output_prefix}/{name}.parquet"
    key = target.removeprefix("s3://staging/")
    s3.delete_object(Bucket="staging", Key=key)
    con.execute(f"COPY (SELECT * FROM {name}_data) TO {sqlstr(target)} (FORMAT PARQUET)")
show_table(["Результат", "Путь в SeaweedFS"],
           [(name, f"{output_prefix}/{name}.parquet") for name in ("staging", "quarantine")])

Результат,Путь в SeaweedFS
staging,s3://staging/seoul-bike-sharing-demand/source_ingested_on=2026-09-30/staging.parquet
quarantine,s3://staging/seoul-bike-sharing-demand/source_ingested_on=2026-09-30/quarantine.parquet


## 5. Проверка результатов

Повторное чтение обоих Parquet-файлов из S3, проверка баланса строк, критических нарушений и признаков качества.

In [16]:
for name in ("staging", "quarantine"):
    path = f"{output_prefix}/{name}.parquet"
    con.execute(f"CREATE OR REPLACE TEMP VIEW saved_{name} AS "
                f"SELECT * FROM read_parquet({sqlstr(path)})")

saved_staging_count = con.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
saved_quarantine_count = con.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]
critical_remaining = con.execute("""
    SELECT count(*) FROM saved_staging
    WHERE event_date IS NULL
       OR hour_of_day IS NULL OR hour_of_day NOT BETWEEN 0 AND 23
       OR rented_bike_count IS NULL OR rented_bike_count < 0
       OR temperature_c IS NULL OR NOT ISFINITE(temperature_c)
       OR season IS NULL OR season NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
       OR season IS DISTINCT FROM CASE
            WHEN MONTH(event_date) IN (12, 1, 2) THEN 'Winter'
            WHEN MONTH(event_date) IN (3, 4, 5) THEN 'Spring'
            WHEN MONTH(event_date) IN (6, 7, 8) THEN 'Summer'
            ELSE 'Autumn' END
       OR functioning_day IS NULL OR functioning_day NOT IN ('Yes', 'No')
""").fetchone()[0]
duplicate_staging = con.execute("""
 SELECT count(*) FROM (
   SELECT event_date, hour_of_day FROM saved_staging
   GROUP BY 1, 2 HAVING count(*) > 1
 )
""").fetchone()[0]
flag_mismatches = con.execute("""
 SELECT count(*) FROM saved_staging
 WHERE quality_non_functioning_hour_violated IS DISTINCT FROM (functioning_day = 'No')
""").fetchone()[0]
flagged = con.execute("""
 SELECT count(*) FROM saved_staging WHERE quality_non_functioning_hour_violated
""").fetchone()[0]
missing_reasons = con.execute("""
 SELECT count(*) FROM saved_quarantine
 WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
    OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]

assert raw_count == saved_staging_count + saved_quarantine_count, "Баланс строк не сошёлся"
assert critical_remaining == 0, "В staging остались критические ошибки"
assert duplicate_staging == 0, "В staging остались дубли бизнес-ключа"
assert flag_mismatches == 0, "Признак нерабочего часа ошибочен"
assert missing_reasons == 0, "В карантине есть строки без объяснения"
run_summary = (raw_count, saved_staging_count, saved_quarantine_count,
               flagged, critical_remaining, duplicate_staging)
show_table(["Raw", "Staging", "Карантин", "Флагов", "Критических", "Дублирующихся ключей"],
           [run_summary])
show_table(["Проверка", "Ошибок"],
           [("Несовпадение флага", flag_mismatches), ("Пустые причины в карантине", missing_reasons)])
show_table(["Дата", "Час", "Флаг", "Статус"], con.execute("""
   SELECT event_date, hour_of_day, quality_non_functioning_hour_violated, functioning_day
   FROM saved_staging WHERE quality_non_functioning_hour_violated LIMIT 2
""").fetchall())
show_table(["Дата из CSV", "Час из CSV", "Правила", "Причины"], con.execute("""
   SELECT "Date", "Hour", quality_rule_ids, quarantine_reason
   FROM saved_quarantine LIMIT 2
""").fetchall())

Raw,Staging,Карантин,Флагов,Критических,Дублирующихся ключей
8760,8760,0,295,0,0


Проверка,Ошибок
Несовпадение флага,0
Пустые причины в карантине,0


Дата,Час,Флаг,Статус
2018-04-11,8,True,No
2018-04-11,13,True,No


Дата из CSV,Час из CSV,Правила,Причины
